In [ ]:
# @title 1. ler KML to coordenadas_acesso.csv

import os
import glob
import re
import pandas as pd

# 1. Definir a pasta onde estão os arquivos KML
pasta_kml = '/content/acesso'

# Buscar todos os arquivos .kml ou .KML na pasta (ignora maiúsculas/minúsculas)
arquivos_kml = glob.glob(os.path.join(pasta_kml, '*.[kK][mM][lL]'))

print(f"🔍 Encontrados {len(arquivos_kml)} arquivos KML na pasta '{pasta_kml}'.\n")

dados_todos_acessos = []

# 2. Ler cada arquivo e extrair as coordenadas
for caminho in arquivos_kml:
    # Pega o nome do arquivo sem a extensão
    nome_acesso = os.path.basename(caminho).replace('.kml', '').replace('.KML', '')
    print(f"⚙️ Extraindo coordenadas de: {nome_acesso}...")

    try:
        with open(caminho, 'r', encoding='utf-8') as f:
            conteudo = f.read()

        # Procura o bloco de coordenadas dentro do KML
        match = re.search(r'<coordinates>(.*?)</coordinates>', conteudo, re.DOTALL)

        if match:
            pontos = match.group(1).strip().split()
            for p in pontos:
                partes = p.split(',')
                if len(partes) >= 2:
                    dados_todos_acessos.append({
                        'Acesso': nome_acesso,
                        'Longitude': float(partes[0]),
                        'Latitude': float(partes[1])
                    })
        else:
            print(f"  -> ⚠️ Aviso: Nenhuma coordenada encontrada em {nome_acesso}.")

    except Exception as e:
        print(f"  -> ❌ Erro ao ler {nome_acesso}: {e}")

# 3. Converter para Tabela (DataFrame) e salvar como CSV Único
if dados_todos_acessos:
    df_coordenadas = pd.DataFrame(dados_todos_acessos)

    # Caminho onde o CSV será salvo
    caminho_csv = '/content/coordenadas_acessos.csv'
    df_coordenadas.to_csv(caminho_csv, index=False)

    print(f"\n✅ Sucesso! Arquivo gerado com {len(df_coordenadas)} pontos no total.")
    print(f"📂 Salvo em: {caminho_csv}\n")

    # Exibe as 5 primeiras linhas na tela para você conferir
    display(df_coordenadas.head())
else:
    print("\n❌ Nenhum dado foi extraído. Verifique se os KMLs estão na pasta correta.")

🔍 Encontrados 11 arquivos KML na pasta '/content/acesso'.

⚙️ Extraindo coordenadas de: AC-2...
⚙️ Extraindo coordenadas de: AC-4b...
⚙️ Extraindo coordenadas de: AC-ADM...
⚙️ Extraindo coordenadas de: AC-4...
⚙️ Extraindo coordenadas de: AC-1...
⚙️ Extraindo coordenadas de: Paiol 1...
⚙️ Extraindo coordenadas de: AC-3...
⚙️ Extraindo coordenadas de: Acesso Principal...
⚙️ Extraindo coordenadas de: AC-EST...
⚙️ Extraindo coordenadas de: Acesso Principal 1...
⚙️ Extraindo coordenadas de: AC-4a...

✅ Sucesso! Arquivo gerado com 1203 pontos no total.
📂 Salvo em: /content/coordenadas_acessos.csv



,Acesso,Longitude,Latitude
0,AC-2,-50.777101,-26.504758
1,AC-2,-50.777478,-26.504690
2,AC-2,-50.777698,-26.504635
3,AC-2,-50.777840,-26.504549
4,AC-2,-50.778089,-26.504499


In [ ]:
# @title 2. Incluir Altura  to coordenadas_acesso.csv
import pandas as pd
import requests
import time
import os

# 1. Carregar o CSV gerado na Etapa 1
caminho_csv_entrada = '/content/coordenadas_acessos.csv'

if not os.path.exists(caminho_csv_entrada):
    print(f"❌ Arquivo '{caminho_csv_entrada}' não encontrado. Rode a Etapa 1 primeiro!")
else:
    df_coords = pd.read_csv(caminho_csv_entrada)
    print(f"📊 Arquivo carregado! Total de coordenadas a processar: {len(df_coords)}\n")

    # Lista para guardar todas as altitudes que a API retornar
    todas_altitudes = []
    tamanho_do_lote = 50  # Manda 50 pontos por vez para não estourar o limite do link

    print("⏳ Iniciando a busca de altitudes (isso pode levar alguns minutos dependendo da quantidade de pontos)...")

    # 2. Busca em lotes com proteção contra o Erro 429
    for i in range(0, len(df_coords), tamanho_do_lote):
        lote = df_coords.iloc[i : i + tamanho_do_lote]

        # Converte as coordenadas para string para colocar no link
        lats = [str(lat) for lat in lote['Latitude']]
        lons = [str(lon) for lon in lote['Longitude']]

        url = f"https://api.open-meteo.com/v1/elevation?latitude={','.join(lats)}&longitude={','.join(lons)}"

        sucesso = False
        tentativas = 0

        while not sucesso and tentativas < 5:
            try:
                resposta = requests.get(url)

                if resposta.status_code == 200:
                    altitudes_lote = resposta.json().get('elevation', [])
                    todas_altitudes.extend(altitudes_lote)
                    sucesso = True

                    # Mostra o progresso na tela
                    progresso = min(i + tamanho_do_lote, len(df_coords))
                    print(f"  -> Processados: {progresso} de {len(df_coords)} pontos...")

                    time.sleep(1.0) # Pausa estratégica para não sobrecarregar a API

                elif resposta.status_code == 429:
                    tentativas += 1
                    tempo_espera = 5 * tentativas
                    print(f"    ⚠️ API sobrecarregada (Erro 429). Pausando por {tempo_espera}s...")
                    time.sleep(tempo_espera)
                else:
                    print(f"    ❌ Erro na API (Código: {resposta.status_code}). Tentando novamente...")
                    tentativas += 1
                    time.sleep(2)

            except Exception as e:
                print(f"    ❌ Erro de conexão: {e}. Tentando novamente...")
                tentativas += 1
                time.sleep(2)

        if not sucesso:
             print("\n❌ Falha na API após 5 tentativas. O processo foi interrompido.")
             break

    # 3. Verificar se buscou tudo e salvar o novo CSV
    if len(todas_altitudes) == len(df_coords):
        df_coords['Altitude (m)'] = todas_altitudes

        caminho_csv_saida = '/content/coordenadas_acessos_altura.csv'
        df_coords.to_csv(caminho_csv_saida, index=False)

        print(f"\n✅ SUCESSO! Todas as {len(df_coords)} altitudes foram encontradas.")
        print(f"📂 Novo arquivo salvo em: {caminho_csv_saida}\n")

        # Mostra as primeiras linhas para conferência
        display(df_coords.head())
    else:
        print(f"\n⚠️ Inconsistência: Encontramos {len(todas_altitudes)} altitudes para {len(df_coords)} pontos.")
        print("Tente rodar novamente em alguns minutos.")

📊 Arquivo carregado! Total de coordenadas a processar: 1203

⏳ Iniciando a busca de altitudes (isso pode levar alguns minutos dependendo da quantidade de pontos)...
  -> Processados: 50 de 1203 pontos...
  -> Processados: 100 de 1203 pontos...
  -> Processados: 150 de 1203 pontos...
  -> Processados: 200 de 1203 pontos...
  -> Processados: 250 de 1203 pontos...
  -> Processados: 300 de 1203 pontos...
  -> Processados: 350 de 1203 pontos...
  -> Processados: 400 de 1203 pontos...
  -> Processados: 450 de 1203 pontos...
  -> Processados: 500 de 1203 pontos...
  -> Processados: 550 de 1203 pontos...
  -> Processados: 600 de 1203 pontos...
  -> Processados: 650 de 1203 pontos...
  -> Processados: 700 de 1203 pontos...
  -> Processados: 750 de 1203 pontos...
  -> Processados: 800 de 1203 pontos...
  -> Processados: 850 de 1203 pontos...
  -> Processados: 900 de 1203 pontos...
    ⚠️ API sobrecarregada (Erro 429). Pausando por 5s...
    ⚠️ API sobrecarregada (Erro 429). Pausando por 10s...
 

,Acesso,Longitude,Latitude,Altitude (m)
0,AC-2,-50.777101,-26.504758,875.0
1,AC-2,-50.777478,-26.504690,875.0
2,AC-2,-50.777698,-26.504635,875.0
3,AC-2,-50.777840,-26.504549,875.0
4,AC-2,-50.778089,-26.504499,875.0


In [ ]:
# @title 3. Calcular VOlumes {"form-width":"10px"}

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
import os
import locale

# ==========================================
# 1. MOTOR DE CÁLCULO V10 - REVISÃO FINAL DE ENGENHARIA
# ==========================================
def projetar_acesso_v10_final(df_coords, nome_projeto, largura_pista=6.0, incl_max=0.16, alt_max=10.0):
    # Cálculo de distâncias e estaqueamento
    lats, lons = np.radians(df_coords['Latitude'].values), np.radians(df_coords['Longitude'].values)
    a = np.sin(np.diff(lats)/2)**2 + np.cos(lats[:-1]) * np.cos(lats[1:]) * np.sin(np.diff(lons)/2)**2
    dists = np.concatenate(([0], 6371000.0 * 2 * np.arcsin(np.sqrt(a))))
    x_total = np.cumsum(dists)

    x_u, idx = np.unique(x_total, return_index=True)
    z_u = df_coords['Altitude (m)'].values[idx]
    f_interp = interp1d(x_u, z_u, kind='linear', fill_value="extrapolate")

    x_est = np.arange(0, x_u[-1], 20.0)
    if x_est[-1] < x_u[-1]: x_est = np.append(x_est, x_u[-1])

    z_est = f_interp(x_est).astype(float)
    N = len(x_est)
    dx = np.diff(x_est)

    def criar_greide(offset):
        g = z_est + offset
        # Trava de altura (H <= 10m) aplicada antes e depois da rampa
        g = np.clip(g, z_est - alt_max, z_est + alt_max)
        for _ in range(3):
            for i in range(1, N):
                d = x_est[i] - x_est[i-1]
                g[i] = np.clip(g[i], g[i-1] - (incl_max*d), g[i-1] + (incl_max*d))
            for i in range(N-2, -1, -1):
                d = x_est[i+1] - x_est[i]
                g[i] = np.clip(g[i], g[i+1] - (incl_max*d), g[i+1] + (incl_max*d))
        # Garantia final do limite de 10m (Prioridade 1)
        return np.clip(g, z_est - alt_max, z_est + alt_max)

    def get_vols(g):
        h = g - z_est
        hc, ha = np.maximum(-h, 0), np.maximum(h, 0)
        ac = (largura_pista * hc) + (1.0 * hc**2)
        aa = (largura_pista * ha) + (1.5 * ha**2)
        v_c = (ac[:-1] + ac[1:]) * 0.5 * dx
        v_a = (aa[:-1] + aa[1:]) * 0.5 * dx
        return v_c, v_a, hc, ha

    # Otimização
    res = minimize_scalar(lambda z: np.sum(get_vols(criar_greide(z))[:2]),
                          bounds=(-alt_max, alt_max), method='bounded')

    g_f = criar_greide(res.x)
    vc_s, va_s, hc, ha = get_vols(g_f)

    vc_col, va_col = np.zeros(N), np.zeros(N)
    vc_col[1:], va_col[1:] = vc_s, va_s

    df_p = pd.DataFrame({
        'Estaca': [f"{int(x//20)}+{x%20:05.2f}".replace('.', ',') for x in x_est],
        'Z_Terreno': np.round(z_est, 2),
        'Z_Greide': np.round(g_f, 2),
        'H_Corte': np.round(hc, 2),
        'H_Aterro': np.round(ha, 2),
        'Vol_Corte_m3': np.round(vc_col, 2),
        'Vol_Aterro_m3': np.round(va_col, 2)
    })

    # --- GRÁFICOS ---
    # Gráfico H (Alturas)
    fig_h = go.Figure()
    fig_h.add_trace(go.Bar(x=df_p['Estaca'], y=df_p['H_Corte'], name='Corte (m)', marker_color='#d9534f'))
    fig_h.add_trace(go.Bar(x=df_p['Estaca'], y=-df_p['H_Aterro'], name='Aterro (m)', marker_color='#0275d8'))
    fig_h.update_layout(title=f"Alturas de Escavação/Aterro (Limite 10m) - {nome_projeto}", barmode='relative', height=300, template='plotly_white')

    # Gráfico Perfil
    fig_p = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])
    fig_p.add_trace(go.Scatter(x=df_p['Estaca'], y=df_p['Z_Terreno'], name='Terreno', line=dict(color='#8B4513', dash='dash')), row=1, col=1)
    fig_p.add_trace(go.Scatter(x=df_p['Estaca'], y=df_p['Z_Greide'], name='Greide', line=dict(color='black', width=3)), row=1, col=1)

    # Pintura das áreas
    fig_p.add_trace(go.Scatter(x=df_p['Estaca'], y=np.minimum(df_p['Z_Terreno'], df_p['Z_Greide']), line=dict(width=0), showlegend=False), row=1, col=1)
    fig_p.add_trace(go.Scatter(x=df_p['Estaca'], y=df_p['Z_Greide'], fill='tonexty', fillcolor='rgba(2, 117, 216, 0.25)', name='Aterro'), row=1, col=1)
    fig_p.add_trace(go.Scatter(x=df_p['Estaca'], y=np.maximum(df_p['Z_Terreno'], df_p['Z_Greide']), fill='tonexty', fillcolor='rgba(217, 83, 79, 0.25)', name='Corte'), row=1, col=1)

    massa = np.cumsum((df_p['Vol_Corte_m3'] * 1.125) - df_p['Vol_Aterro_m3'])
    fig_p.add_trace(go.Scatter(x=df_p['Estaca'], y=massa, name='Balanço de Massa', fill='tozeroy', line=dict(color='purple')), row=2, col=1)
    fig_p.update_layout(height=500, template='plotly_white')

    html_fig = pio.to_html(fig_h, full_html=False, include_plotlyjs=False) + pio.to_html(fig_p, full_html=False, include_plotlyjs=False)

    return np.sum(vc_s), np.sum(va_s), df_p, html_fig

# ==========================================
# 2. GERAÇÃO DO HTML COM FORMATAÇÃO BR
# ==========================================
csv_path = '/content/coordenadas_acessos_altura.csv'
if os.path.exists(csv_path):
    df_all = pd.read_csv(csv_path)
    html_content = ""
    resumo_vols = []

    for n in df_all['Acesso'].unique():
        print(f"▶️ Processando: {n}")
        df_sub = df_all[df_all['Acesso'] == n].reset_index(drop=True)
        try:
            vc, va, d_est, g_html = projetar_acesso_v10_final(df_sub, n)
            saldo = (vc * 1.125) - va
            tipo = "Bota-fora" if saldo >= 0 else "Empréstimo"

            resumo_vols.append({'Acesso': n, 'Corte (m3)': vc, 'Aterro (m3)': va, 'Saldo (m3)': abs(saldo), 'Tipo': tipo})

            # Formatação de milhar para a tabela detalhada (exibição apenas)
            d_est_formatada = d_est.copy()
            for col in ['Z_Terreno', 'Z_Greide', 'Vol_Corte_m3', 'Vol_Aterro_m3']:
                d_est_formatada[col] = d_est_formatada[col].apply(lambda x: f"{x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

            html_content += f"""
            <div class='box'>
                <h2 class='border-bottom pb-2'>Acesso: {n}</h2>
                {g_html}
                <details class='mt-4'>
                    <summary class='btn btn-sm btn-outline-dark'>Exibir Planilha de Estacas Detalhada</summary>
                    <div class='table-responsive mt-3'>
                        {d_est_formatada.to_html(classes='table table-sm table-striped text-end', index=False)}
                    </div>
                </details>
            </div>
            """
        except Exception as e:
            print(f"❌ Erro em {n}: {e}")

    if resumo_vols:
        df_res = pd.DataFrame(resumo_vols)
        tc, ta = df_res['Corte (m3)'].sum(), df_res['Aterro (m3)'].sum()
        saldo_geral = (tc * 1.125) - ta
        tipo_geral = "BOTA-FORA" if saldo_geral >= 0 else "EMPRÉSTIMO"
        cor_geral = "success" if saldo_geral >= 0 else "danger"

        # Formatação de milhar para o resumo
        def fmt_br(x): return f"{x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

        df_res_fmt = df_res.copy()
        for col in ['Corte (m3)', 'Aterro (m3)', 'Saldo (m3)']:
            df_res_fmt[col] = df_res_fmt[col].apply(fmt_br)

        final_report = f"""
        <!DOCTYPE html><html><head>
        <meta charset="UTF-8"><title>Relatório Terraplenagem V10</title>
        <link href='https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css' rel='stylesheet'>
        <script src='https://cdn.plot.ly/plotly-2.24.1.min.js'></script>
        <style>
            body{{background:#f0f2f5; padding:30px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;}}
            .box{{background:#fff; padding:30px; margin-bottom:60px; border-radius:15px; box-shadow:0 8px 16px rgba(0,0,0,0.08); border: 1px solid #e1e4e8;}}
            summary{{ font-weight: bold; padding: 10px; border-radius: 5px; }}
            .card-summary{{ border: none; border-radius: 15px; color: white; transition: 0.3s; }}
            .card-summary:hover{{ transform: translateY(-5px); }}
        </style>
        </head><body class='container-fluid' style='max-width: 1400px;'>
            <h1 class='text-center fw-bold mb-5'>PAINEL DE TERRAPLENAGEM (V10 - EXECUTIVO)</h1>
            <div class='row mb-5 g-4'>
                <div class='col-md-4'><div class='card card-summary p-4 bg-secondary'><h6>VOLUME TOTAL CORTE</h6><h2 class='fw-bold'>{fmt_br(tc)} m3</h2></div></div>
                <div class='col-md-4'><div class='card card-resumo p-4 bg-primary text-white card-summary'><h6>VOLUME TOTAL ATERRO</h6><h2 class='fw-bold'>{fmt_br(ta)} m3</h2></div></div>
                <div class='col-md-4'><div class='card card-summary p-4 bg-{cor_geral}'><h6>BALANÇO GERAL ({tipo_geral})</h6><h2 class='fw-bold'>{fmt_br(abs(saldo_geral))} m3</h2></div></div>
            </div>
            <div class='box'><h4 class='fw-bold mb-4'>Resumo Consolidado por Trecho</h4>{df_res_fmt.to_html(classes='table table-hover text-center', index=False)}</div>
            {html_content}
        </body></html>
        """
        with open('/content/relatorio_terraplenagem_V10.html', 'w', encoding='utf-8') as f: f.write(final_report)
        from google.colab import files
        files.download('/content/relatorio_terraplenagem_V10.html')

▶️ Processando: AC-2
▶️ Processando: AC-4b
▶️ Processando: AC-ADM
▶️ Processando: AC-4
▶️ Processando: AC-1
▶️ Processando: Paiol 1
▶️ Processando: AC-3
▶️ Processando: Acesso Principal
▶️ Processando: AC-EST
▶️ Processando: Acesso Principal 1
▶️ Processando: AC-4a


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>